# Titanic Survival Analysis (Exploratory Data Analysis)
**Red & White Skill Education | Data Analysis Project**

## Objective
Perform exploratory data analysis (EDA) on the Titanic passenger dataset to understand **which factors influenced passenger survival**, and visualize survival rates by **class, gender, age** and other features.

## Tools / Libraries
Pandas (data handling), NumPy (numerical work), Matplotlib and Seaborn (visualization).

## Dataset columns
| Column | Description |
|--------|-------------|
| PassengerId | Unique passenger number |
| Survived | 0 = did not survive, 1 = survived |
| Pclass | Ticket class (1 = upper, 2 = middle, 3 = lower) |
| Name | Passenger name (contains the title, e.g. Mr, Mrs) |
| Sex | male / female |
| Age | Age in years |
| SibSp | Number of siblings / spouses aboard |
| Parch | Number of parents / children aboard |
| Ticket | Ticket number |
| Fare | Ticket price |
| Cabin | Cabin number |
| Embarked | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |

> **Note:** The notebook works with `titanic.csv` in this folder. The same code also runs on the original Kaggle `train.csv` (just keep it in this folder; it is picked up automatically if `titanic.csv` is not found).

## 1. Import Libraries

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

# Folder where all charts are saved (used in README.md)
os.makedirs("images", exist_ok=True)

SURVIVAL_COLORS = {0: "#d9534f", 1: "#5cb85c"}   # red = died, green = survived


def save_fig(name):
    """Save the current figure into the images/ folder."""
    plt.tight_layout()
    plt.savefig(f"images/{name}.png", dpi=120, bbox_inches="tight")

print("Libraries imported successfully.")

## 2. Load the Dataset

In [ ]:
DATA_PATH = None
for candidate in ("titanic.csv", "train.csv"):
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break

if DATA_PATH is None:
    raise FileNotFoundError("Place titanic.csv (or Kaggle's train.csv) in the same folder as this notebook.")

df = pd.read_csv(DATA_PATH)
print(f"Loaded '{DATA_PATH}' -> {df.shape[0]} rows and {df.shape[1]} columns")
display(df.head())

## 3. Understand the Data

In [ ]:
print("Shape:", df.shape)
print()
df.info()

In [ ]:
print("Statistical summary of numeric columns:")
display(df.describe().round(2))

print("\nSummary of text columns:")
display(df.describe(exclude="number"))

## 4. Missing Values and Duplicates
`Age`, `Cabin` and `Embarked` usually contain missing values. We first measure how much is missing, then decide how to handle each column.

In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage (%)": (df.isnull().mean() * 100).round(2),
})
missing = missing[missing["Missing Values"] > 0].sort_values("Missing Values", ascending=False)
display(missing)

print("Duplicate rows:", df.duplicated().sum())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(missing.index, missing["Percentage (%)"], color="#c0504d")
for bar, value in zip(bars, missing["Percentage (%)"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 1, f"{value:.1f}%", ha="center", fontweight="bold")
ax.set_title("Percentage of Missing Values per Column")
ax.set_ylabel("Missing (%)")
ax.set_ylim(0, max(missing["Percentage (%)"].max() + 12, 20))
save_fig("01_missing_values")
plt.show()

## 5. Data Cleaning and Feature Engineering

| Step | Decision |
|------|----------|
| `Age` | Fill missing values with the **median age of the passenger's class and gender** (more accurate than one global value) |
| `Embarked` | Fill missing values with the **most frequent port** (mode) |
| `Cabin` | ~77% missing, so it is **not imputed**. We create `Deck` (first letter of cabin, "Unknown" if missing) and `HasCabin` |
| `Title` | Extracted from `Name` (Mr, Mrs, Miss, Master, Rare) |
| `FamilySize` | `SibSp + Parch + 1` |
| `IsAlone` | 1 if `FamilySize == 1` |
| `AgeGroup`, `FareBand` | Grouped versions of `Age` and `Fare` for easy comparison |

The original DataFrame `df` is kept untouched; all changes are made on a copy called `df_clean`.

In [ ]:
df_clean = df.copy()

# --- Title from Name -------------------------------------------------------
df_clean["Title"] = df_clean["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
df_clean["Title"] = df_clean["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
common_titles = ["Mr", "Mrs", "Miss", "Master"]
df_clean["Title"] = df_clean["Title"].where(df_clean["Title"].isin(common_titles), "Rare")

# --- Age: median of the same Pclass + Sex group ----------------------------
group_median = df_clean.groupby(["Pclass", "Sex"])["Age"].transform("median")
df_clean["Age"] = df_clean["Age"].fillna(group_median)
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())      # safety net

# --- Embarked: most frequent port -------------------------------------------
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])

# --- Fare: safety net (normally no missing values) ---------------------------
df_clean["Fare"] = df_clean["Fare"].fillna(df_clean.groupby("Pclass")["Fare"].transform("median"))

# --- Cabin -> Deck and HasCabin ---------------------------------------------
df_clean["HasCabin"] = df_clean["Cabin"].notna().astype(int)
df_clean["Deck"] = df_clean["Cabin"].str[0].fillna("Unknown")

# --- Family features ---------------------------------------------------------
df_clean["FamilySize"] = df_clean["SibSp"] + df_clean["Parch"] + 1
df_clean["IsAlone"] = (df_clean["FamilySize"] == 1).astype(int)
df_clean["FamilySizeGroup"] = pd.cut(
    df_clean["FamilySize"], bins=[0, 1, 4, 100], labels=["Alone", "Small (2-4)", "Large (5+)"]
)

# --- Age groups and fare bands ----------------------------------------------
df_clean["AgeGroup"] = pd.cut(
    df_clean["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child (0-12)", "Teen (13-18)", "Young Adult (19-35)", "Adult (36-60)", "Senior (60+)"],
)
# rank(method="first") avoids duplicate bin edges when many fares are equal
df_clean["FareBand"] = pd.qcut(
    df_clean["Fare"].rank(method="first"), 4, labels=["Low", "Medium", "High", "Very High"]
)

# --- Readable label for charts ----------------------------------------------
df_clean["Outcome"] = df_clean["Survived"].map({0: "Did not survive", 1: "Survived"})

print("Missing values after cleaning (important columns):")
print(df_clean[["Age", "Embarked", "Fare", "Title", "Deck", "AgeGroup", "FareBand"]].isnull().sum())
display(df_clean[["Name", "Title", "Age", "AgeGroup", "Fare", "FareBand", "FamilySize", "IsAlone", "Deck"]].head())

## 6. Univariate Analysis (one variable at a time)

### 6.1 Overall survival

In [ ]:
counts = df_clean["Survived"].value_counts().sort_index()      # 0 first, then 1
overall_rate = df_clean["Survived"].mean() * 100

print(f"Passengers who survived     : {counts.get(1, 0)}  ({overall_rate:.1f}%)")
print(f"Passengers who did not live : {counts.get(0, 0)}  ({100 - overall_rate:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].bar(["Did not survive", "Survived"], [counts.get(0, 0), counts.get(1, 0)],
            color=[SURVIVAL_COLORS[0], SURVIVAL_COLORS[1]])
for i, value in enumerate([counts.get(0, 0), counts.get(1, 0)]):
    axes[0].text(i, value + 8, str(value), ha="center", fontweight="bold")
axes[0].set_title("Survival Count")
axes[0].set_ylabel("Passengers")

axes[1].pie([counts.get(0, 0), counts.get(1, 0)], labels=["Did not survive", "Survived"], autopct="%1.1f%%",
            colors=[SURVIVAL_COLORS[0], SURVIVAL_COLORS[1]], startangle=90, wedgeprops={"edgecolor": "white"})
axes[1].set_title("Survival Share")

save_fig("02_overall_survival")
plt.show()

### 6.2 Who was on board? (passenger profile)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, column, title in zip(
    axes.ravel(),
    ["Pclass", "Sex", "Embarked", "FamilySizeGroup"],
    ["Passengers by Class", "Passengers by Gender", "Passengers by Port of Embarkation", "Passengers by Family Size"],
):
    values = df_clean[column].astype(str).value_counts()
    if column == "Pclass":
        values = values.sort_index()
    bars = ax.bar(values.index, values.values, color=sns.color_palette("deep", len(values)))
    for bar, value in zip(bars, values.values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 5, str(value), ha="center", fontweight="bold")
    ax.set_title(title)
    ax.set_ylabel("Passengers")

save_fig("03_passenger_profile")
plt.show()

### 6.3 Distribution of Age and Fare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(df_clean["Age"], bins=30, kde=True, color="#4c72b0", ax=axes[0])
axes[0].axvline(df_clean["Age"].median(), color="red", linestyle="--", label=f"Median = {df_clean['Age'].median():.0f}")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age (years)")
axes[0].legend()

sns.histplot(df_clean["Fare"], bins=40, kde=True, color="#55a868", ax=axes[1])
axes[1].set_title("Fare Distribution (right-skewed)")
axes[1].set_xlabel("Fare")

save_fig("04_age_fare_distribution")
plt.show()

print(f"Average age : {df_clean['Age'].mean():.1f} years")
print(f"Average fare: {df_clean['Fare'].mean():.2f}   |   Median fare: {df_clean['Fare'].median():.2f}")

## 7. Survival Rate by Individual Factors
A helper function draws a bar chart of the **survival rate (%)** for any column, with a dashed line showing the overall survival rate, and returns a summary table.

In [ ]:
def survival_rate_plot(column, title, order=None, file_name=None, rotation=0):
    """Bar chart of survival rate (%) per category of `column` + summary table."""
    grouped = df_clean.groupby(column, observed=False)["Survived"].agg(["count", "mean"])
    grouped["mean"] = grouped["mean"] * 100
    grouped.index = grouped.index.astype(str)
    if order is not None:
        grouped = grouped.reindex([str(o) for o in order if str(o) in grouped.index])
    grouped = grouped.dropna()

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(grouped.index, grouped["mean"], color=sns.color_palette("deep", len(grouped)))
    ax.axhline(overall_rate, color="gray", linestyle="--", label=f"Overall survival ({overall_rate:.1f}%)")
    for bar, value, n in zip(bars, grouped["mean"], grouped["count"]):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 1.5, f"{value:.1f}%\n(n={int(n)})",
                ha="center", fontweight="bold", fontsize=9)
    ax.set_ylim(0, min(max(grouped["mean"].max() + 20, 30), 125))
    ax.set_ylabel("Survival rate (%)")
    ax.set_title(title)
    ax.legend(loc="upper right")
    plt.xticks(rotation=rotation)
    if file_name:
        save_fig(file_name)
    plt.show()

    table = grouped.rename(columns={"count": "Passengers", "mean": "Survival Rate (%)"}).round(2)
    display(table)
    return table

### 7.1 Survival by Gender

In [ ]:
sex_table = survival_rate_plot("Sex", "Survival Rate by Gender", order=["female", "male"], file_name="05_survival_by_gender")

### 7.2 Survival by Passenger Class

In [ ]:
class_table = survival_rate_plot("Pclass", "Survival Rate by Passenger Class", order=[1, 2, 3], file_name="06_survival_by_class")

### 7.3 Survival by Age Group

In [ ]:
age_order = ["Child (0-12)", "Teen (13-18)", "Young Adult (19-35)", "Adult (36-60)", "Senior (60+)"]
age_table = survival_rate_plot("AgeGroup", "Survival Rate by Age Group", order=age_order,
                               file_name="07_survival_by_age_group", rotation=15)

### 7.4 Survival by Port of Embarkation

In [ ]:
embark_table = survival_rate_plot("Embarked", "Survival Rate by Port of Embarkation", order=["C", "Q", "S"],
                                  file_name="08_survival_by_embarked")

### 7.5 Survival by Family Size

In [ ]:
family_table = survival_rate_plot("FamilySizeGroup", "Survival Rate by Family Size",
                                  order=["Alone", "Small (2-4)", "Large (5+)"], file_name="09_survival_by_family")

### 7.6 Survival by Title

In [ ]:
title_table = survival_rate_plot("Title", "Survival Rate by Title", order=["Mr", "Mrs", "Miss", "Master", "Rare"],
                                 file_name="10_survival_by_title")

### 7.7 Survival by Fare Band and Deck

In [ ]:
fare_table = survival_rate_plot("FareBand", "Survival Rate by Fare Band (quartiles)",
                                order=["Low", "Medium", "High", "Very High"], file_name="11_survival_by_fare")

In [ ]:
deck_table = survival_rate_plot("Deck", "Survival Rate by Deck (Unknown = no cabin recorded)",
                                order=["A", "B", "C", "D", "E", "F", "G", "T", "Unknown"], file_name="12_survival_by_deck")

## 8. Multivariate Analysis (combining factors)

### 8.1 Class and Gender together

In [ ]:
class_sex = df_clean.pivot_table(index="Pclass", columns="Sex", values="Survived", aggfunc="mean") * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

class_sex.plot(kind="bar", ax=axes[0], color=["#e377c2", "#1f77b4"], rot=0)
axes[0].set_title("Survival Rate by Class and Gender")
axes[0].set_xlabel("Passenger Class")
axes[0].set_ylabel("Survival rate (%)")
axes[0].set_ylim(0, 115)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.1f%%", padding=2, fontsize=9)

sns.heatmap(class_sex.T, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={"label": "Survival rate (%)"}, ax=axes[1])
axes[1].set_title("Heatmap: Gender vs Class")
axes[1].set_xlabel("Passenger Class")
axes[1].set_ylabel("")

save_fig("13_class_gender")
plt.show()
display(class_sex.round(2))

### 8.2 Gender and Age Group together

In [ ]:
sex_age = df_clean.pivot_table(index="Sex", columns="AgeGroup", values="Survived", aggfunc="mean", observed=False) * 100
sex_age = sex_age[[a for a in age_order if a in sex_age.columns]]

plt.figure(figsize=(9, 3.8))
sns.heatmap(sex_age, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100, cbar_kws={"label": "Survival rate (%)"})
plt.title("Survival Rate (%) by Gender and Age Group")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=15)
save_fig("14_gender_age_heatmap")
plt.show()

### 8.3 Age distribution of survivors vs non-survivors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df_clean, x="Age", hue="Survived", bins=30, kde=True, element="step",
             palette=SURVIVAL_COLORS, ax=axes[0])
axes[0].set_title("Age Distribution by Survival (0 = died, 1 = survived)")

sns.boxplot(data=df_clean, x="Pclass", y="Age", hue="Survived", palette=SURVIVAL_COLORS, ax=axes[1])
axes[1].set_title("Age by Class and Survival")
axes[1].set_xlabel("Passenger Class")

save_fig("15_age_survival")
plt.show()

### 8.4 Fare, class and survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df_clean, x="Pclass", y="Fare", hue="Survived", palette=SURVIVAL_COLORS, ax=axes[0])
axes[0].set_yscale("symlog")
axes[0].set_title("Fare by Class and Survival (log scale)")
axes[0].set_xlabel("Passenger Class")

port_class = pd.crosstab(df_clean["Embarked"], df_clean["Pclass"], normalize="index") * 100
port_class.plot(kind="bar", stacked=True, ax=axes[1], rot=0, color=["#4c72b0", "#dd8452", "#55a868"])
axes[1].set_title("Class Mix at Each Port of Embarkation (%)")
axes[1].set_xlabel("Port")
axes[1].set_ylabel("Share of passengers (%)")
axes[1].legend(title="Class")

save_fig("16_fare_port_class")
plt.show()

### 8.5 Correlation analysis

In [ ]:
corr_data = df_clean[["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "HasCabin"]].copy()
corr_data["Sex (female=1)"] = (df_clean["Sex"] == "female").astype(int)
corr = corr_data.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.4, 1]})

sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.8}, ax=axes[0])
axes[0].set_title("Correlation Heatmap")

survival_corr = corr["Survived"].drop("Survived").sort_values()
axes[1].barh(survival_corr.index, survival_corr.values,
             color=["#d9534f" if v < 0 else "#5cb85c" for v in survival_corr.values])
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Correlation of Each Feature with Survival")
axes[1].set_xlabel("Correlation coefficient")

save_fig("17_correlation")
plt.show()
display(survival_corr.round(3).to_frame("Correlation with Survived"))

## 9. Key Insights (calculated from the data)

In [ ]:
def rate_of(mask):
    return df_clean.loc[mask, "Survived"].mean() * 100

female_rate = rate_of(df_clean["Sex"] == "female")
male_rate = rate_of(df_clean["Sex"] == "male")
class_rates = {c: rate_of(df_clean["Pclass"] == c) for c in (1, 2, 3)}
child_rate = rate_of(df_clean["Age"] <= 12)
adult_rate = rate_of(df_clean["Age"] > 12)
senior_rate = rate_of(df_clean["Age"] > 60)
alone_rate = rate_of(df_clean["IsAlone"] == 1)
family_rate = rate_of(df_clean["IsAlone"] == 0)
cabin_rate = rate_of(df_clean["HasCabin"] == 1)
no_cabin_rate = rate_of(df_clean["HasCabin"] == 0)
port_rates = df_clean.groupby("Embarked")["Survived"].mean().mul(100)
best_group = class_sex.stack().idxmax()
worst_group = class_sex.stack().idxmin()

print(f"1. Overall survival rate               : {overall_rate:.1f}%")
print(f"2. Women vs men                        : {female_rate:.1f}% vs {male_rate:.1f}%")
print(f"3. 1st / 2nd / 3rd class               : {class_rates[1]:.1f}% / {class_rates[2]:.1f}% / {class_rates[3]:.1f}%")
print(f"4. Children (<=12) vs older passengers : {child_rate:.1f}% vs {adult_rate:.1f}%")
print(f"5. Passengers aged 60+                 : {senior_rate:.1f}%")
print(f"6. Travelling alone vs with family     : {alone_rate:.1f}% vs {family_rate:.1f}%")
print(f"7. With cabin record vs without        : {cabin_rate:.1f}% vs {no_cabin_rate:.1f}%")
print(f"8. Best port for survival              : {port_rates.idxmax()} ({port_rates.max():.1f}%)")
print(f"9. Highest-survival group              : class {best_group[0]} {best_group[1]}s ({class_sex.stack().max():.1f}%)")
print(f"10. Lowest-survival group              : class {worst_group[0]} {worst_group[1]}s ({class_sex.stack().min():.1f}%)")

## 10. Conclusions

* **Gender was the strongest factor.** Women survived at a much higher rate than men, which matches the "women and children first" evacuation rule.
* **Passenger class mattered.** Survival dropped steadily from 1st to 3rd class. Upper-class passengers had cabins closer to the boat deck and better access to lifeboats.
* **Class and gender combined:** women in 1st and 2nd class almost all survived, while men in 2nd and 3rd class had the lowest chances.
* **Age:** age had a weaker effect than gender and class. Children did better than the average adult passenger, while very small age groups (such as 60+) should be read with caution because they contain few passengers.
* **Family size:** passengers travelling in small families (2-4 members) survived more often than people travelling alone, while very large families (5+) had the lowest survival.
* **Fare, deck and cabin** are closely linked to class: higher fares and recorded cabins go with higher survival.
* **Port of embarkation** shows differences that largely reflect the different class mix at each port (see chart 8.4) rather than the port itself.
* **Correlation is not causation.** These patterns describe the passengers in this dataset; several features (class, fare, cabin, deck) overlap with each other.

## 11. Limitations and Future Work
* `Age` had many missing values and was filled with group medians, so age-based results are approximate.
* `Cabin` is mostly missing, so deck results rely on a small subset of passengers.
* Future work: build a prediction model (e.g. logistic regression or random forest) and compare feature importance with the findings above.

## 12. Save the Cleaned Dataset

In [ ]:
df_clean.to_csv("titanic_cleaned.csv", index=False)
print("Cleaned dataset saved as titanic_cleaned.csv  ->", df_clean.shape)
print("Charts saved in the images/ folder:", len(os.listdir("images")), "files")